# 03 — Eksperimen Skenario B, C, D (Proyek TTKI P1)

Notebook ini mengerjakan tiga skenario pembanding setelah baseline (skenario A di notebook 02).
Setiap skenario hanya mengubah **satu hal** dari skenario sebelumnya, supaya perubahan skor bisa dijelaskan penyebabnya.

| Skenario | Pembagian | Fitur | Perbandingan |
|---|---|---|---|
| B | Acak (sama dengan A) | F-URL saja | A→B: pengaruh kelompok fitur |
| C | Registered-domain-disjoint | F-URL saja | B→C: sensitivitas terhadap protokol split |
| D | Registered-domain-disjoint (sama dengan C) | 5 fitur F-URL hasil seleksi | C→D: pengaruh pengurangan fitur |

**Aturan yang dijaga**
- Imputasi, scaling, dan seleksi fitur hanya di-fit pada data latih setiap fold (semuanya di dalam `Pipeline`).
- Test final tidak disentuh (`RUN_FINAL_TEST = False`).
- Seed, parameter, kebijakan PSL, dan daftar fitur diambil dari `configs/experiment_config.json`.
- Notebook 02 harus sudah dijalankan lebih dulu (menghasilkan `split_indices_random.npz` dan baris skenario A).

In [1]:
import os, json, time, warnings
import numpy as np, pandas as pd, tldextract, sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix
warnings.filterwarnings("ignore", category=UserWarning)

CFG_PATH = next(p for p in ["../configs/experiment_config.json", "configs/experiment_config.json"] if os.path.exists(p))
cfg = json.load(open(CFG_PATH))
ROOT = os.path.dirname(os.path.dirname(CFG_PATH)) or "."
RESULTS_DIR = os.path.join(ROOT, "results")
DATA_PATH = next(p for p in [os.path.join(ROOT, "data", cfg["dataset"]["file"]),
                             "/mnt/user-data/uploads/" + cfg["dataset"]["file"]] if os.path.exists(p))
SEED = cfg["seed"]
RUN_FINAL_TEST = False
K_FEATURES = 5
print("sklearn", sklearn.__version__, "| tldextract", tldextract.__version__, "| PSL:", cfg["registered_domain_policy"]["psl"])

sklearn 1.9.1 | tldextract 5.4.0 | PSL: private


## 1. Data dan fitur

In [2]:
df = pd.read_csv(DATA_PATH)
y = (1 - df["label"]).values          # phishing = 1
F_URL = cfg["features"]["F_URL"]
X_URL = df[F_URL].astype(float).values
print(f"{len(df):,} baris | F-URL: {len(F_URL)} fitur | phishing: {y.mean():.1%}")

235,795 baris | F-URL: 18 fitur | phishing: 42.8%


## 2. Registered domain (kebijakan private PSL)
Dengan private PSL, platform hosting seperti `github.io` atau `web.app` diperlakukan sebagai suffix tersendiri,
sehingga `a.github.io` dan `b.github.io` dihitung sebagai dua registered domain berbeda.
Kebijakan ini hanya dipakai untuk split skenario C dan D; skenario A dan B memakai split acak.

In [3]:
use_private = cfg["registered_domain_policy"]["psl"] == "private"
ext = tldextract.TLDExtract(suffix_list_urls=(), include_psl_private_domains=use_private)
def reg_domain(url):
    r = ext(url)
    return f"{r.domain}.{r.suffix}".lower() if r.suffix else r.domain.lower()

groups = np.array([reg_domain(u) for u in df["URL"].tolist()])
print(f"{len(set(groups)):,} registered domain")
print(pd.Series(groups).value_counts().head(5).to_string())

197,709 registered domain
weeblysite.com      3097
ipfs.io             1559
xsph.ru              928
google.com           782
godaddysites.com     781


## 3. Indeks split
Semua indeks di bawah adalah **indeks baris absolut** pada data (bukan relatif terhadap development set).
- **Skenario B** memakai split acak yang sama dengan A (dibaca dari `split_indices_random.npz`).
- **Skenario C dan D** memakai split baru berbasis registered domain; indeksnya disimpan ke `split_indices_grouped.npz`.

In [4]:
sp = np.load(os.path.join(RESULTS_DIR, "split_indices_random.npz"))
folds_random = [(sp[f"fold{i}_train"], sp[f"fold{i}_val"]) for i in range(1, 6)]

idx = np.arange(len(df))
dev_g, test_g = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(idx, y, groups))
rel = StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(np.arange(len(dev_g)), y[dev_g], groups[dev_g])
folds_grouped = [(dev_g[tr], dev_g[va]) for tr, va in rel]      # ubah ke indeks absolut

np.savez(os.path.join(RESULTS_DIR, "split_indices_grouped.npz"), dev=dev_g, test=test_g,
         **{f"fold{i+1}_train": tr for i, (tr, va) in enumerate(folds_grouped)},
         **{f"fold{i+1}_val":   va for i, (tr, va) in enumerate(folds_grouped)})

print(f"Grouped: dev {len(dev_g):,} | test {len(test_g):,} | rasio phishing dev/test: {y[dev_g].mean():.4f}/{y[test_g].mean():.4f}")
print("Domain yang sama di dev dan test:", len(set(groups[dev_g]) & set(groups[test_g])))
print("Domain yang sama antara train dan val (maks. 5 fold):",
      max(len(set(groups[tr]) & set(groups[va])) for tr, va in folds_grouped))

Grouped: dev 188,636 | test 47,159 | rasio phishing dev/test: 0.4281/0.4281
Domain yang sama di dev dan test: 0
Domain yang sama antara train dan val (maks. 5 fold): 0


## 4. Fungsi bantu

In [5]:
def evaluate(y_true, proba, thr=cfg["evaluation"]["threshold"]):
    pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"f1_phishing": f1_score(y_true, pred), "roc_auc": roc_auc_score(y_true, proba),
            "precision": precision_score(y_true, pred), "recall": recall_score(y_true, pred),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def make_pipeline(model_name, k_select=None):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if k_select:
        steps.append(("select", SelectKBest(f_classif, k=k_select)))
    p = cfg["models"][model_name]
    if model_name == "logistic_regression":
        steps += [("scale", StandardScaler()),
                  ("clf", LogisticRegression(penalty=p["penalty"], C=p["C"], solver=p["solver"], max_iter=p["max_iter"]))]
    else:
        steps.append(("clf", RandomForestClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                     max_features=p["max_features"], random_state=p["random_state"], n_jobs=-1)))
    return Pipeline(steps)

def run_scenario(scenario, protocol, fold_pairs, k_select=None):
    rows = []
    for model_name in ["logistic_regression", "random_forest"]:
        for k, (tr, va) in enumerate(fold_pairs, start=1):
            pipe = make_pipeline(model_name, k_select)
            t0 = time.perf_counter(); pipe.fit(X_URL[tr], y[tr]); fit_s = time.perf_counter() - t0
            m = evaluate(y[va], pipe.predict_proba(X_URL[va])[:, 1])
            selected = []
            if k_select:
                mask = pipe.named_steps["select"].get_support()
                selected = [F_URL[i] for i, keep in enumerate(mask) if keep]
            rows.append({"scenario": scenario, "protocol": protocol, "model": model_name, "fold": k,
                         "n_features": k_select or len(F_URL), "n_train": len(tr), "n_val": len(va),
                         "fit_seconds": round(fit_s, 2), "selected_features": "|".join(selected), **m})
            print(f"[{scenario}] {model_name:20s} fold {k}: F1={m['f1_phishing']:.4f}  AUC={m['roc_auc']:.4f}")
    return pd.DataFrame(rows)

## 5. Skenario B — split acak, F-URL saja
Beda dari A hanya pada fitur (F-Page dibuang). Perbandingan A→B mengukur **pengaruh kelompok fitur**.

In [6]:
res_B = run_scenario('B', 'random', folds_random)

/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[B] logistic_regression  fold 1: F1=0.9951  AUC=0.9985


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[B] logistic_regression  fold 2: F1=0.9958  AUC=0.9982


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[B] logistic_regression  fold 3: F1=0.9952  AUC=0.9985


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[B] logistic_regression  fold 4: F1=0.9948  AUC=0.9982


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[B] logistic_regression  fold 5: F1=0.9956  AUC=0.9982
[B] random_forest        fold 1: F1=0.9967  AUC=0.9985
[B] random_forest        fold 2: F1=0.9968  AUC=0.9983
[B] random_forest        fold 3: F1=0.9966  AUC=0.9980
[B] random_forest        fold 4: F1=0.9965  AUC=0.9979
[B] random_forest        fold 5: F1=0.9971  AUC=0.9981


## 6. Skenario C — split per registered domain, F-URL saja
Beda dari B hanya pada protokol split. Perbandingan B→C mengukur **sensitivitas terhadap protokol split**,
bukan otomatis besarnya leakage.

In [7]:
res_C = run_scenario('C', 'grouped', folds_grouped)

/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[C] logistic_regression  fold 1: F1=0.9950  AUC=0.9981


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[C] logistic_regression  fold 2: F1=0.9950  AUC=0.9981


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[C] logistic_regression  fold 3: F1=0.9954  AUC=0.9984


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[C] logistic_regression  fold 4: F1=0.9954  AUC=0.9983


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[C] logistic_regression  fold 5: F1=0.9955  AUC=0.9985
[C] random_forest        fold 1: F1=0.9966  AUC=0.9983
[C] random_forest        fold 2: F1=0.9964  AUC=0.9983
[C] random_forest        fold 3: F1=0.9971  AUC=0.9985
[C] random_forest        fold 4: F1=0.9970  AUC=0.9978
[C] random_forest        fold 5: F1=0.9958  AUC=0.9984


## 7. Skenario D — split sama dengan C, 5 fitur hasil seleksi
Beda dari C hanya pada jumlah fitur. `SelectKBest(f_classif, k=5)` berada **di dalam pipeline**, jadi di-fit
hanya pada data latih tiap fold. Perbandingan C→D mengukur **pengaruh pengurangan fitur**.

In [8]:
res_D = run_scenario('D', 'grouped', folds_grouped, k_select=K_FEATURES)

/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[D] logistic_regression  fold 1: F1=0.9843  AUC=0.9963


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[D] logistic_regression  fold 2: F1=0.9834  AUC=0.9964


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[D] logistic_regression  fold 3: F1=0.9832  AUC=0.9966


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[D] logistic_regression  fold 4: F1=0.9854  AUC=0.9965


/home/zafka/BINDO/Paper/ttki-P1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


[D] logistic_regression  fold 5: F1=0.9833  AUC=0.9971
[D] random_forest        fold 1: F1=0.9938  AUC=0.9970
[D] random_forest        fold 2: F1=0.9929  AUC=0.9972
[D] random_forest        fold 3: F1=0.9938  AUC=0.9974
[D] random_forest        fold 4: F1=0.9944  AUC=0.9973
[D] random_forest        fold 5: F1=0.9932  AUC=0.9975


## 8. Simpan hasil
Baris skenario A dari notebook 02 dipertahankan; baris B, C, D selalu **diganti** (bukan ditambah),
sehingga menjalankan ulang notebook ini tidak menggandakan hasil.

In [9]:
path = os.path.join(RESULTS_DIR, "metrics_per_fold.csv")
prev = pd.read_csv(path)
prev = prev[prev["scenario"] == "A"]
cols = list(prev.columns) + [c for c in res_B.columns if c not in prev.columns]
all_m = pd.concat([prev, res_B, res_C, res_D], ignore_index=True)[cols]
all_m.to_csv(path, index=False)

sel = [{"scenario": "D", "model": r.model, "fold": r.fold, "feature": f}
       for r in res_D.itertuples() for f in r.selected_features.split("|")]
sel_df = pd.DataFrame(sel); sel_df.to_csv(os.path.join(RESULTS_DIR, "selected_features.csv"), index=False)

metrics = ["f1_phishing", "roc_auc", "precision", "recall"]
summ = all_m.groupby(["scenario", "model"])[metrics].agg(["mean", "std"]).round(4)
summ.to_csv(os.path.join(RESULTS_DIR, "all_scenarios_cv_summary.csv"))
print("Baris per skenario:", all_m["scenario"].value_counts().sort_index().to_dict())
print(); print(summ.to_string())
print(); print("Fitur terpilih skenario D (jumlah kemunculan dari 10 pelatihan):")
print(sel_df.groupby("feature").size().sort_values(ascending=False).to_string())

Baris per skenario: {'A': 10, 'B': 10, 'C': 10, 'D': 10}

                             f1_phishing         roc_auc         precision          recall        
                                    mean     std    mean     std      mean     std    mean     std
scenario model                                                                                    
A        logistic_regression      0.9990  0.0001  1.0000  0.0000    0.9993  0.0002  0.9988  0.0002
         random_forest            0.9998  0.0001  1.0000  0.0000    0.9999  0.0001  0.9997  0.0002
B        logistic_regression      0.9953  0.0004  0.9983  0.0001    0.9996  0.0001  0.9911  0.0008
         random_forest            0.9967  0.0002  0.9982  0.0002    0.9988  0.0003  0.9947  0.0004
C        logistic_regression      0.9952  0.0002  0.9983  0.0002    0.9995  0.0001  0.9910  0.0005
         random_forest            0.9966  0.0005  0.9982  0.0003    0.9984  0.0012  0.9947  0.0007
D        logistic_regression      0.9839  0.0009  0

## 9. Test final (dimatikan)
Jalankan **sekali** setelah seluruh rancangan dibekukan dan disepakati dengan Dika dan dosen.

In [10]:
if RUN_FINAL_TEST:
    plan = {"A": (None, "random"), "B": (F_URL, "random"), "C": (F_URL, "grouped"), "D": (F_URL, "grouped")}
    print("Aktifkan dengan sengaja: lengkapi rencana test final di sini setelah rancangan dibekukan.")
else:
    print("Test final belum dijalankan (RUN_FINAL_TEST = False).")

Test final belum dijalankan (RUN_FINAL_TEST = False).
